# Notebook 02 — Agentic RAG: retrieval that knows when it's wrong

**Workshop:** Agentic AI for Scientists — Week 3 (From Loops to Graphs)
**Block:** Agentic RAG (30 min)
**Goal:** Turn Week 2's plain RAG into a **graph that self-corrects**. We add a **relevance grader** after retrieval and a **web-search fallback**, so when the local corpus comes up short the agent notices and goes looking elsewhere — instead of confidently answering from irrelevant chunks.

**Three patterns, one graph.** *Corrective RAG (CRAG)* grades retrieved docs and falls back to web search. *Self-RAG* additionally grades the **generation** (is it grounded? does it answer?). *Adaptive RAG* adds a **router** that picks the strategy per question. We build CRAG end to end, then bolt on the Self-RAG graders and describe the Adaptive router.


## 0. Setup

We use **Gemini embeddings** + an **in-memory vector store** — no Chroma, no sentence-transformers, so none of Week 2's numpy ABI pain. Tavily powers the web-search fallback (optional; a mock runs without a key).


In [ ]:
%pip install -q \
    "langgraph>=1.2.4" "langchain>=1.3.2" "langchain-core>=1.4.0" \
    "langchain-google-genai>=4.2.4" "langchain-community>=0.4" \
    "langchain-text-splitters>=0.3" "beautifulsoup4" "tavily-python>=0.7" python-dotenv

In [ ]:
import os

# Free Gemini API key (no credit card): https://aistudio.google.com/apikey
# Colab: add GOOGLE_API_KEY under the key icon (left sidebar) -> "Secrets".
# Local: put GOOGLE_API_KEY=AIza... in a .env file next to this notebook.
try:
    from google.colab import userdata  # type: ignore
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()

# Accept GEMINI_API_KEY as an alias (some AI Studio users store it under that name).
if not os.environ.get("GOOGLE_API_KEY") and os.environ.get("GEMINI_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"]

assert os.environ.get("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY first (see the comment above)."
print("API key loaded.")


## 1. Build a tiny vector store

We index a small curated corpus on agents/RAG, embed it with Gemini, and store it in memory. **Small on purpose:** Gemini's *free embedding* tier is rate-limited, so embedding a whole blog post at once trips a 429 — and a classroom all embedding at once would too. A handful of focused documents demonstrates the agentic-RAG graph identically. (Want a bigger, real corpus? Flip `USE_WEB = True` below — just mind the free-tier limit.)


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# A compact, information-rich corpus — enough for retrieval + grading to be meaningful.
CORPUS = [
    "An LLM agent is a language model wrapped in a control loop with memory, tools, and a planner. "
    "It decides an action, runs it, observes the result, and repeats until the task is done.",
    "ReAct (Reasoning + Acting) interleaves a 'Thought' (reasoning) with an 'Action' (a tool call) "
    "and an 'Observation' (the tool's result), looping until a final answer. It was introduced by Yao et al., 2022.",
    "Reflection lets an agent critique its own output and revise it. Reflexion extends this by storing "
    "the critique in memory so each revision addresses prior weaknesses, converging instead of wandering.",
    "Retrieval-augmented generation (RAG) grounds an LLM in retrieved documents: a retriever fetches "
    "relevant chunks by embedding similarity, and the model conditions its answer on them to reduce hallucination.",
    "Corrective RAG (CRAG) grades retrieved documents for relevance and falls back to web search when "
    "the local corpus is weak. Self-RAG additionally grades the generation for grounding and usefulness.",
    "Tool use means giving the model functions it can call — a calculator, a search API, a database query. "
    "The model emits a structured tool call; your code runs the tool and feeds the result back.",
    "Embeddings map text to vectors so similar meanings sit close together. A vector store indexes these "
    "vectors for fast nearest-neighbour search; hybrid retrieval mixes dense vectors with sparse BM25 scoring.",
    "LangGraph models an agent as a stateful graph: nodes are steps, edges are control flow, and a typed "
    "state is threaded through. Conditional edges enable branching and cycles a plain loop can't express cleanly.",
]
docs = [Document(page_content=t) for t in CORPUS]

USE_WEB = False   # flip to True to load real web pages instead (heavier on the free embedding tier)
if USE_WEB:
    try:
        from langchain_community.document_loaders import WebBaseLoader
        loaded = WebBaseLoader(["https://lilianweng.github.io/posts/2023-06-23-agent/"]).load()
        web = [d for d in loaded if d.page_content.strip()]
        if web:
            docs = web
            print(f"Loaded {len(web)} page(s) from the web.")
    except Exception as e:
        print("Web load failed (%s) -> keeping the inline corpus." % type(e).__name__)

splits = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=80).split_documents(docs)
splits = splits[:24]   # safety cap for the free embedding tier
print(f"{len(splits)} chunks.")


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

# gemini-embedding-001 is the free embedding model (3072-dim). The free tier is
# rate-limited, so we embed in small batches with a short backoff on a 429.
import time
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")


def build_store(chunks, batch=4, pause=2.0):
    store = InMemoryVectorStore(embeddings)
    for i in range(0, len(chunks), batch):
        for attempt in range(4):
            try:
                store.add_documents(chunks[i:i + batch])
                break
            except Exception as e:
                if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                    wait = pause * (attempt + 1)
                    print(f"  rate-limited on batch {i}; waiting {wait:.0f}s...")
                    time.sleep(wait)
                else:
                    raise
        time.sleep(pause)   # gentle pacing for the free tier
    return store


vectorstore = build_store(splits)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
hits = retriever.invoke("What is ReAct?")
print(f"retrieved {len(hits)} chunks; first 120 chars:\n", hits[0].page_content[:120])


## 2. The GraphState

The state threads the question, the retrieved docs, the running generation, and a `web_search` flag the grader sets.


In [ ]:
from typing import TypedDict, List
from langchain_core.documents import Document


class RagState(TypedDict):
    question: str
    documents: List[Document]
    generation: str
    web_search: str        # "Yes"/"No" — set by the grader, read by the router


## 3. Nodes: retrieve → grade → (web_search) → generate

The **grader** is the agentic part: a structured yes/no per document. If any doc is irrelevant, we flag a web search.


In [ ]:
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)


class GradeDoc(BaseModel):
    binary_score: str = Field(description="'yes' if the document is relevant to the question, else 'no'.")


doc_grader = llm.with_structured_output(GradeDoc)


def retrieve(state: RagState) -> dict:
    return {"documents": retriever.invoke(state["question"])}


def grade_documents(state: RagState) -> dict:
    kept, need_search = [], "No"
    for d in state["documents"]:
        g = doc_grader.invoke(
            f"Question: {state['question']}\n\nDocument: {d.page_content}\n\n"
            "Is this document relevant to the question? Answer 'yes' or 'no'."
        )
        if g.binary_score.strip().lower() == "yes":
            kept.append(d)
        else:
            need_search = "Yes"     # at least one weak doc -> fall back to web
    return {"documents": kept, "web_search": need_search}


def generate(state: RagState) -> dict:
    context = "\n\n".join(d.page_content for d in state["documents"]) or "(no documents)"
    ans = llm.invoke(
        f"Answer the question using ONLY the context. If the context is insufficient, say so.\n\n"
        f"Context:\n{context}\n\nQuestion: {state['question']}"
    ).content
    return {"generation": ans}


## 4. The web-search fallback (Tavily, with a mock)

When the grader flags weak retrieval, this node fetches fresh results and appends them as a document. Without a `TAVILY_API_KEY` it returns a clearly-labelled mock so the graph still runs.


In [ ]:
def _tavily_search(query: str) -> str:
    key = _secret_tavily()
    if not key:
        return f"[MOCK web result for '{query}': add TAVILY_API_KEY for real search.]"
    from tavily import TavilyClient
    # include_answer=True gives a clean synthesized answer; we keep the raw snippets
    # too, and tag the block with the query so the (strict) generate step knows what
    # these results are about.
    res = TavilyClient(api_key=key).search(
        query, max_results=5, search_depth="advanced", include_answer="advanced"
    )
    # Tavily's include_answer returns a clean synthesized result; prefer it. Raw
    # snippets from different sites are often partial/conflicting and confuse the
    # strict generate step. Fall back to snippets only if there is no answer. The
    # query tag tells the generate step what these results are about.
    if res.get("answer"):
        return f"Web search results for: {query}\n\n{res['answer']}"
    snippets = "\n\n".join(r["content"] for r in res.get("results", []))
    return f"Web search results for: {query}\n\n{snippets}"


def _secret_tavily():
    import os
    try:
        from google.colab import userdata  # type: ignore
        v = userdata.get("TAVILY_API_KEY")
        if v:
            return v
    except Exception:
        pass
    return os.environ.get("TAVILY_API_KEY")


def web_search(state: RagState) -> dict:
    text = _tavily_search(state["question"])
    docs = state["documents"] + [Document(page_content=text)]
    return {"documents": docs}


def decide_to_generate(state: RagState) -> str:
    return "web_search" if state["web_search"] == "Yes" else "generate"


## 5. Assemble the CRAG graph

`retrieve → grade_documents → {web_search → generate | generate}`. The conditional edge is the correction.


In [ ]:
from langgraph.graph import StateGraph, START, END

crag = StateGraph(RagState)
crag.add_node("retrieve", retrieve)
crag.add_node("grade_documents", grade_documents)
crag.add_node("web_search", web_search)
crag.add_node("generate", generate)

crag.add_edge(START, "retrieve")
crag.add_edge("retrieve", "grade_documents")
crag.add_conditional_edges(
    "grade_documents", decide_to_generate,
    {"web_search": "web_search", "generate": "generate"},
)
crag.add_edge("web_search", "generate")
crag.add_edge("generate", END)

crag_app = crag.compile()


**See the graph.** This renders the corrective-RAG control flow: retrieve → grade → (web-search the gaps?) → generate.

In [ ]:
# --- See the graph this code builds: Corrective RAG ---
# A compiled LangGraph app can draw its own topology: nodes are steps,
# edges are control flow. This picture IS the agent you just wired.
try:
    from IPython.display import Image, display
    display(Image(crag_app.get_graph().draw_mermaid_png()))
except Exception:
    # draw_mermaid_png needs network/graphviz; ASCII always works
    print(crag_app.get_graph().draw_ascii())

## Run it: the graph in action, judged against ground truth

We ask two questions and print a small **ground-truth** reference under each answer, so you can judge the agent the way an evaluator would (Week 5).

- The first is about **ReAct the agent pattern** (disambiguated from React the JavaScript library), which our corpus covers.
- The second, the **2026 Winter Olympics medal totals**, is not in the corpus at all.

Watch `web_search`. The grader is deliberately strict: if any retrieved chunk looks weak it sends the graph to the web, so you will often see `Yes` even when the corpus does have something relevant. That is the corrective step doing its job. And note the second answer can still come back thin even after a web search, which is exactly why a ground-truth reference (and real evaluation) matters.

In [ ]:
# A tiny "ground truth" set so you can judge each answer against reality.
# Comparing an answer to a known reference is exactly what evaluation is (Week 5).
GROUND_TRUTH = {
    "react": (
        "ReAct (Yao et al., 2022, arXiv:2210.03629) is an LLM-agent pattern that interleaves "
        "Thought (reasoning) -> Action (a tool call) -> Observation in a loop until a final answer. "
        "It is NOT React, the JavaScript UI library."
    ),
    "olympics": (
        "2026 Winter Olympics (Milan-Cortina) final medal table, top totals as Gold/Silver/Bronze (Total): "
        "Norway 18/12/11 (41), United States 12/12/9 (33), Netherlands 10/7/3 (20), Italy 10/6/14 (30), "
        "Germany 8/10/8 (26), France 8/9/6 (23). 29 NOCs won at least one medal. "
        "Source: https://en.wikipedia.org/wiki/2026_Winter_Olympics_medal_table"
    ),
}

DEMO = [
    # in the corpus, so it is answered locally (web_search stays "No")
    ("What is the ReAct agent pattern in LLMs, i.e. reasoning and acting, not the React JS library?", "react"),
    # not in the corpus, so the grader flags weak retrieval and the graph goes to the web
    ("What were the total medal counts by country at the 2026 Winter Olympics?", "olympics"),
]

for q, key in DEMO:
    print("\n" + "=" * 70 + f"\nQ: {q}")
    out = crag_app.invoke({"question": q, "documents": [], "generation": "", "web_search": "No"})
    print("web_search used:", out["web_search"])   # No = local corpus, Yes = went to the web
    print("answer:      ", out["generation"][:400])
    print("ground truth:", GROUND_TRUTH[key])

The first question is in-corpus → grader passes → straight to generate. The second isn't → grader flags it → web-search fallback fires first. That self-correction is the whole point of *agentic* RAG.

## 6. Self-RAG — also grade the *generation*

CRAG grades retrieval. **Self-RAG** adds two more graders after `generate`: is the answer **grounded** in the docs (no hallucination), and does it **answer the question**? Each is a structured yes/no that routes back to regenerate or re-search. Here's the grader + the routing logic you'd insert after `generate`.


In [ ]:
class GradeGen(BaseModel):
    grounded: str = Field(description="'yes' if the answer is supported by the documents, else 'no'.")
    answers: str = Field(description="'yes' if the answer addresses the question, else 'no'.")


gen_grader = llm.with_structured_output(GradeGen)


def grade_generation(state: RagState) -> str:
    context = "\n\n".join(d.page_content for d in state["documents"])
    g = gen_grader.invoke(
        f"Documents:\n{context}\n\nQuestion: {state['question']}\n\nAnswer: {state['generation']}\n\n"
        "Is the answer grounded in the documents, and does it answer the question?"
    )
    if g.grounded.strip().lower() != "yes":
        return "not_grounded"     # -> loop back to generate (hallucination)
    if g.answers.strip().lower() != "yes":
        return "not_useful"       # -> loop back to web_search / transform query
    return "useful"               # -> END


# demo on the last answer
print(grade_generation(out))


**Adaptive RAG** is the third point on this graph: add a **router node at the entry** that classifies the question first — direct-answer (no retrieval), vectorstore, or web-search — and sends it down the right path. Same primitives (a conditional edge off `START`), one more decision. The LangGraph docs' Adaptive-RAG tutorial wires all three together; you now have every piece.

---

## Recap

- **CRAG** = retrieve → **grade docs** → web-search fallback → generate. The grader makes it agentic.
- **Self-RAG** = also grade the generation for **grounding** and **usefulness**, looping until both pass.
- **Adaptive RAG** = a front **router** that picks the retrieval strategy per question.
- Gemini embeddings + `InMemoryVectorStore` keep it dependency-light and Colab-safe.

Next: **production** — give an agent memory across turns and the ability to pause for a human.
